In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_hineng_train.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/pos_hineng_validation.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/pos_spaeng_train.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/sa_spaeng_validation.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_msaea_test.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_spaeng_test.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/ner_spaeng_test.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_spaeng_train.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_hineng_validation.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_nepeng_test.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/ner_hineng_train.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/ner_msaea_validation.csv
/kaggle/input/datasets/jennyjingjing/switchlingua/LinCE/lid_nep

In [2]:
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/

In [3]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

Tue Oct 13 06:55:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q vllm
!python -c "import vllm, torch; print(vllm.__version__, torch.__version__, torch.cuda.is_available())"

0.31.0 2.13.0+cu130 True


In [5]:
import subprocess, time, requests
server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct-AWQ",
     "--dtype", "float16", "--quantization", "awq",
     "--max-model-len", "2048", "--max-num-seqs", "32",
     "--gpu-memory-utilization", "0.85",
     "--enforce-eager", "--enable-prefix-caching", "--port", "8000"],
    stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)

for i in range(60):                      # 最多等 10 分钟
    if server.poll() is not None:
        print("服务已退出，看日志"); break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print(f"就绪，用时约 {i*10}s"); break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(10)

服务已退出，看日志


In [6]:
!pip uninstall -y torchaudio
!python -c "import transformers.audio_utils; print('ok')"

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
ok


In [7]:
import subprocess, time, requests
server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct-AWQ",
     "--dtype", "float16", "--quantization", "awq",
     "--max-model-len", "2048", "--max-num-seqs", "32",
     "--gpu-memory-utilization", "0.85",
     "--enforce-eager", "--enable-prefix-caching", "--port", "8000"],
    stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT)

for i in range(60):                      # 最多等 10 分钟
    if server.poll() is not None:
        print("服务已退出，看日志"); break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print(f"就绪，用时约 {i*10}s"); break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(10)

就绪，用时约 120s


In [8]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    messages=[{"role": "user", "content": "你好，用一句话介绍你自己"}],
    max_tokens=50, temperature=0)
print(r.choices[0].message.content)

我是Qwen，由阿里云开发的AI助手，很高兴与您交流。


In [9]:
print(r.choices[0].finish_reason, r.usage.completion_tokens)

stop 18


In [10]:
import glob, os
csvs = sorted(glob.glob("/kaggle/input/**/SwitchLingua/**/*.csv", recursive=True))
for p in csvs:
    print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")
DATA_DIR = os.path.dirname(csvs[0])
print("\nDATA_DIR =", DATA_DIR, "| 共", len(csvs), "个文件")

    55.7 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Arabic_eng.csv
   694.5 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Can_Eng.csv
    56.7 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Chinese_eng.csv
    51.7 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/French_eng.csv
    44.5 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/German_Eng.csv
    55.6 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/German_French.csv
    56.3 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Hindi_eng.csv
    51.9 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Italian_eng.csv
    56.7 MB  /kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text/Japanese_eng.csv
    52.4 MB  /ka

In [11]:
# %% [1] 安装依赖（每个 session 运行一次）
!pip install -q wordfreq jieba

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 10.9 MB/s eta 0:00:00


In [12]:
# %% [2] 配置
import os, re, ast, time, asyncio
import numpy as np, pandas as pd
 
DATA_DIR    = "/kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text"
OUT_DIR     = "/kaggle/working/cs_eval"
MODEL       = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # 换成组里确定的 Qwen 版本，需和 vllm serve 一致
N_SENT      = 1000     # 每个 CSV 抽多少句
SEED        = 42
CONCURRENCY = 32       # 和 vllm serve 的 --max-num-seqs 保持一致
MIN_SCORE   = None     # 例如 7.0：只保留 LinguaMaster 评分 >= 7 的样本
os.makedirs(OUT_DIR, exist_ok=True)
 
# 文件名 -> (第一语言, 第二语言, 标注方式, wordfreq 代码1, 代码2)
# script：两种语言文字系统不同，按 Unicode 文字判断，标签基本准确
# wordfreq：两种语言都用拉丁字母，按词频判断，标签带噪声
LANGS = {
    "Arabic_eng.csv":      ("Arabic",     "English", "script",   None,  None),
    "Can_Eng.csv":         ("Cantonese",  "English", "script",   None,  None),
    "Chinese_eng.csv":     ("Chinese",    "English", "script",   None,  None),
    "Hindi_eng.csv":       ("Hindi",      "English", "script",   None,  None),
    "Japanese_eng.csv":    ("Japanese",   "English", "script",   None,  None),
    "Korean_eng.csv":      ("Korean",     "English", "script",   None,  None),
    "Russian_eng.csv":     ("Russian",    "English", "script",   None,  None),
    "French_eng.csv":      ("French",     "English", "wordfreq", "fr",  "en"),
    "German_Eng.csv":      ("German",     "English", "wordfreq", "de",  "en"),
    "German_French.csv":   ("German",     "French",  "wordfreq", "de",  "fr"),
    "Italian_eng.csv":     ("Italian",    "English", "wordfreq", "it",  "en"),
    "Spanish_eng.csv":     ("Spanish",    "English", "wordfreq", "es",  "en"),
    "Malay_eng.csv":       ("Malay",      "English", "wordfreq", "ms",  "en"),
    "Philippines_eng.csv": ("Filipino",   "English", "wordfreq", "fil", "en"),
    "Vietnamese_eng.csv":  ("Vietnamese", "English", "wordfreq", "vi",  "en"),
}
 
RUN_FILES = ["Chinese_eng.csv"]   # 这次只跑这些文件；全部跑时改成 RUN_FILES = list(LANGS)

In [13]:
print(DATA_DIR, N_SENT, RUN_FILES)

/kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text 1000 ['Chinese_eng.csv']


In [14]:
# %% [3] 读取、抽样、分词、打标签
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="jieba")

import jieba
from wordfreq import zipf_frequency, available_languages
jieba.setLogLevel(60)
 
_avail = available_languages()
for f, (_, _, mode, c1, c2) in LANGS.items():
    if mode == "wordfreq":
        for c in (c1, c2):
            if c not in _avail:
                print(f"警告：wordfreq 不支持 '{c}'（{f}），该文件的标签不可靠")
 
def extract_text(row):
    """从 data_generation_result 取出句子；该列缺失时从 summary 里解析。"""
    raw = row.get("data_generation_result")
    if not isinstance(raw, str) or not raw.strip():
        m = re.search(r"data_generation_result:\s*(\[.*?\])\s*\n", str(row.get("summary", "")), re.S)
        raw = m.group(1) if m else None
    if raw is None:
        return None
    try:
        val = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return raw.strip()
    if isinstance(val, list):
        return " ".join(v for v in val if isinstance(v, str)).strip() or None
    return str(val)
 
def load_and_sample(fname):
    path = os.path.join(DATA_DIR, fname)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    want = ["conversation_type", "score", "data_generation_result"]
    if "data_generation_result" not in cols:
        want.append("summary")          # 没有文本列时才读体积很大的 summary 列
    df = pd.read_csv(path, usecols=[c for c in want if c in cols])
    if "conversation_type" in df:
        df = df[df["conversation_type"] == "single_turn"]
    if MIN_SCORE is not None and "score" in df:
        df = df[pd.to_numeric(df["score"], errors="coerce") >= MIN_SCORE]
    df = df.assign(text=df.apply(extract_text, axis=1)).dropna(subset=["text"])
    s = df.sample(n=min(N_SENT, len(df)), random_state=SEED)
    return s.reset_index().rename(columns={"index": "row_id"})[["row_id", "text"]]
 
LAT   = r"[A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+(?:['’\-][A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+)*"
CJK   = r"[\u3400-\u4dbf\u4e00-\u9fff]+"
OTHER = r"[\u3040-\u309f]+|[\u30a0-\u30ff]+|[\uac00-\ud7af]+|[\u0400-\u04ff]+|[\u0600-\u06ff]+|[\u0900-\u097f]+"
TOKEN_RE = re.compile(f"(?P<lat>{LAT})|(?P<cjk>{CJK})|(?P<oth>{OTHER})")
 
def tokenize(text, l1):
    """返回 [(词, 在原文中的起始位置, 是否拉丁字母)]；中文/粤语用 jieba 切词，数字和标点不算词。"""
    out = []
    for m in TOKEN_RE.finditer(text):
        if m.lastgroup == "cjk" and l1 in ("Chinese", "Cantonese"):
            for w, s, _ in jieba.tokenize(m.group()):
                out.append((w, m.start() + s, False))
        else:
            out.append((m.group(), m.start(), m.lastgroup == "lat"))
    return out
 
def _zipf(w, code):
    try:
        return zipf_frequency(w, code)
    except LookupError:
        return 0.0
 
def tag_tokens(toks, mode, c1, c2):
    """给每个词标 L1 / L2。词频难以区分时沿用上一个词的语言。"""
    tags, prev = [], "L1"
    for w, _, is_lat in toks:
        if mode == "script":
            t = "L2" if is_lat else "L1"
        else:
            f1, f2 = _zipf(w.lower(), c1), _zipf(w.lower(), c2)
            t = prev if abs(f1 - f2) < 0.5 else ("L1" if f1 > f2 else "L2")
        tags.append(t)
        prev = t
    return tags
 
def build_examples(sample, fname):
    """每句话的每个词边界生成一个预测样本：给定前缀，下一个词是否切换语言。"""
    l1, l2, mode, c1, c2 = LANGS[fname]
    rows = []
    for _, r in sample.iterrows():
        toks = tokenize(r.text, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        for i in range(1, len(toks)):
            rows.append(dict(file=fname, row_id=r.row_id, pos=i,
                             prefix=r.text[:toks[i][1]].rstrip(),
                             last_word=toks[i - 1][0], next_word=toks[i][0],
                             label=int(tags[i] != tags[i - 1]), l1=l1, l2=l2))
    return pd.DataFrame(rows)

In [15]:
# %% [4] 标签检查：正式跑之前，先看几句话的切词和标注是否合理
def show_labels(fname, k=2):
    l1, l2, mode, c1, c2 = LANGS[fname]
    for t in load_and_sample(fname).head(k).text:
        toks = tokenize(t, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        print(" ".join(f"{w}/{g}" for (w, _, _), g in zip(toks, tags)), "\n")
 
for f in RUN_FILES:
    print("====", f)
    show_labels(f, k=3)
# 跑印地语时注意：如果是拉丁字母转写，这里会全部标成 L2，需要改用 wordfreq 方式

==== Chinese_eng.csv
我/L1 觉得/L1 下周/L1 港股/L1 应该/L1 会/L1 有/L1 反弹/L1 especially/L2 if/L2 the/L2 tech/L2 stocks/L2 continue/L2 to/L2 perform/L2 well/L2 like/L2 today/L2 市场/L1 情绪/L1 慢慢/L1 回暖/L1 maybe/L2 we/L2 will/L2 see/L2 more/L2 investors/L2 coming/L2 back/L2 into/L2 the/L2 market/L2 我/L1 打算/L1 密切/L1 关注/L1 金融/L1 板块/L1 因为/L1 他们/L1 最近/L1 表现/L1 得/L1 quite/L2 strong/L2 

论坛/L1 现场/L1 气氛/L1 非常/L1 热烈/L1 Many/L2 experts/L2 shared/L2 their/L2 insights/L2 on/L2 the/L2 future/L2 of/L2 information/L2 technology/L2 与会/L1 嘉宾/L1 对/L1 深化/L1 行业/L1 应用/L1 表示/L1 高度/L1 关注/L1 They/L2 believe/L2 collaboration/L2 will/L2 drive/L2 innovation/L2 in/L2 the/L2 sector/L2 中国/L1 电子/L1 信息产业/L1 发展/L1 研究院/L1 的/L1 刘文强/L1 书记/L1 发表/L1 了/L1 精彩/L1 演讲/L1 His/L2 speech/L2 emphasized/L2 the/L2 importance/L2 of/L2 new/L2 quality/L2 productivity/L2 本次/L1 论坛/L1 汇聚/L1 了/L1 多方/L1 力量/L1 共同/L1 探讨/L1 行业/L1 发展/L1 This/L2 event/L2 reflects/L2 the/L2 nation's/L2 commitment/L2 to/L2 technological/L2 progress/L2 

最近/L1 苹果/L1 的/L1 股票/L1 又/L1

In [16]:
# %% [5] 调用模型：只生成 1 个 token，用 Yes/No 的 logprob 算切换概率
from openai import AsyncOpenAI
aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none")
 
SYSTEM = ("You are analyzing code-switched text that is being typed word by word. "
          "Given the text so far, predict whether the NEXT word will be in a different "
          "language from the LAST word. Answer with exactly one word: Yes or No.")
 
def make_messages(ex):
    # 固定内容在前、前缀在后，同一句话的相邻请求共享前缀，prefix caching 才能生效
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Languages: {ex['l1']} and {ex['l2']}\n"
                                        f"Text so far: {ex['prefix']}\n"
                                        f"Last word: {ex['last_word']}\n"
                                        f"Will the next word switch language?"}]
 
def p_yes_from(top):
    py = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("yes"))
    pn = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("no"))
    return py / (py + pn) if py + pn > 0 else np.nan
 
async def predict_one(ex, sem):
    async with sem:
        t0 = time.perf_counter()
        try:
            r = await aclient.chat.completions.create(
                model=MODEL, messages=make_messages(ex),
                max_tokens=1, temperature=0, logprobs=True, top_logprobs=10,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}})
            p, err = p_yes_from(r.choices[0].logprobs.content[0].top_logprobs), None
        except Exception as e:
            p, err = np.nan, repr(e)[:200]
        # 只生成 1 个 token，这个延迟基本等于 TTFT
        return dict(p_yes=p, latency=time.perf_counter() - t0, error=err)
 
async def run_examples(exs):
    sem = asyncio.Semaphore(CONCURRENCY)
    t0 = time.perf_counter()
    res = await asyncio.gather(*(predict_one(e, sem) for e in exs.to_dict("records")))
    wall = time.perf_counter() - t0
    return pd.concat([exs.reset_index(drop=True), pd.DataFrame(res)], axis=1), wall

In [17]:
# %% [6] 主循环：每个 CSV 单独保存结果，session 断了重新运行会跳过已完成的文件
# 注意：Notebook 里直接用 await，不要用 asyncio.run()
for fname in RUN_FILES:
    out_path = os.path.join(OUT_DIR, fname.replace(".csv", "_pred.csv"))
    if os.path.exists(out_path):
        print("已完成，跳过", fname); continue
    if not os.path.exists(os.path.join(DATA_DIR, fname)):
        print("找不到文件，跳过", fname); continue
    sample = load_and_sample(fname)
    exs = build_examples(sample, fname)
    print(f"{fname}: {len(sample)} 句，{len(exs)} 个预测位置，切换比例 {exs.label.mean():.1%}")
    res, wall = await run_examples(exs)
    res["wall_time"] = wall
    res.to_csv(out_path, index=False)
    print(f"  用时 {wall:.1f}s，吞吐 {len(exs) / wall:.1f} req/s，失败 {res.error.notna().sum()} 个")

Chinese_eng.csv: 1000 句，54712 个预测位置，切换比例 10.8%
  用时 1217.1s，吞吐 45.0 req/s，失败 0 个


In [18]:
# %% [7] 汇总：准确率 + 句子级 bootstrap 置信区间 + 基线 + 切换类 F1 + 延迟
def bootstrap_acc(df, B=1000, seed=0):
    # 同一句话里的预测不独立，所以按句子重抽样
    g = df.groupby("row_id")["correct"].agg(["sum", "count"]).values
    idx = np.random.default_rng(seed).integers(0, len(g), size=(B, len(g)))
    accs = g[idx, 0].sum(1) / g[idx, 1].sum(1)
    return np.percentile(accs, [2.5, 97.5])
 
rows = []
for f in sorted(os.listdir(OUT_DIR)):
    if not f.endswith("_pred.csv"):
        continue
    d = pd.read_csv(os.path.join(OUT_DIR, f)).dropna(subset=["p_yes"])
    if d.empty:
        continue
    d["pred"] = (d.p_yes > 0.5).astype(int)
    d["correct"] = (d.pred == d.label).astype(int)
    tp = int(((d.pred == 1) & (d.label == 1)).sum())
    prec, rec = tp / max(d.pred.sum(), 1), tp / max(d.label.sum(), 1)
    lo, hi = bootstrap_acc(d)
    rows.append(dict(
        dataset=f.replace("_pred.csv", ""), n_sent=d.row_id.nunique(), n_pos=len(d),
        switch_rate=d.label.mean(),
        acc=d.correct.mean(), ci_low=lo, ci_high=hi,
        baseline_acc=1 - d.label.mean(),          # 永远猜"不切换"的准确率
        switch_precision=prec, switch_recall=rec,
        switch_f1=2 * prec * rec / max(prec + rec, 1e-9),
        lat_p50=d.latency.quantile(.50), lat_p95=d.latency.quantile(.95),
        lat_p99=d.latency.quantile(.99), throughput=len(d) / d.wall_time.iloc[0]))
 
summary = pd.DataFrame(rows).round(4)
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)
summary

,dataset,n_sent,n_pos,switch_rate,acc,ci_low,ci_high,baseline_acc,switch_precision,switch_recall,switch_f1,lat_p50,lat_p95,lat_p99,throughput
0,Chinese_eng,1000,54712,0.1078,0.4544,0.4437,0.4668,0.8922,0.1111,0.5799,0.1865,0.7018,0.7517,0.7742,44.9512


In [19]:
from sklearn.metrics import roc_auc_score, average_precision_score
d = pd.read_csv(f"{OUT_DIR}/Chinese_eng_pred.csv").dropna(subset=["p_yes"])
print("p_yes 分布:", d.p_yes.describe().round(3).to_dict())
print(f"AUROC = {roc_auc_score(d.label, d.p_yes):.3f}  (0.5 = 随机)")
print(f"AUPRC = {average_precision_score(d.label, d.p_yes):.3f}  (随机 = {d.label.mean():.3f})")

# 一半句子选阈值，另一半句子评估，避免在测试集上调参
ids = d.row_id.unique(); rng = np.random.default_rng(0); rng.shuffle(ids)
dev, test = d[d.row_id.isin(ids[:len(ids)//2])], d[d.row_id.isin(ids[len(ids)//2:])]
ths = np.linspace(0, 1, 101)
best = ths[np.argmax([((dev.p_yes > t) == dev.label).mean() for t in ths])]
print(f"最佳阈值 {best:.2f}，测试集准确率 {((test.p_yes > best) == test.label).mean():.3f}")

# 前缀以标点结尾（句子边界）vs 句子中间
d["at_boundary"] = d.prefix.str.strip().str[-1].isin(list("。！？.!?，,；;"))
print(d.groupby("at_boundary").agg(n=("label", "size"), switch_rate=("label", "mean"),
                                   auroc=("p_yes", lambda s: roc_auc_score(d.loc[s.index, "label"], s))).round(3))

p_yes 分布: {'count': 54712.0, 'mean': 0.558, 'std': 0.41, 'min': 0.0, '25%': 0.083, '50%': 0.672, '75%': 0.981, 'max': 1.0}
AUROC = 0.506  (0.5 = 随机)
AUPRC = 0.106  (随机 = 0.108)
最佳阈值 1.00，测试集准确率 0.892
                 n  switch_rate  auroc
at_boundary                           
False        47761        0.032  0.466
True          6951        0.631  0.505


In [20]:
print(d.groupby(pd.cut(d.pos, [0, 5, 10, 20, 40, 200])).p_yes.mean().round(3))

pos
(0, 5]       0.520
(5, 10]      0.372
(10, 20]     0.542
(20, 40]     0.577
(40, 200]    0.612
Name: p_yes, dtype: float64


/tmp/ipykernel_23/4254069820.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(d.groupby(pd.cut(d.pos, [0, 5, 10, 20, 40, 200])).p_yes.mean().round(3))


In [21]:
from sklearn.metrics import roc_auc_score
LAT_RE, CJK_RE = re.compile(LAT), re.compile(CJK)

async def lm_one(ex, sem):
    async with sem:
        r = await aclient.completions.create(model=MODEL, prompt=ex["prefix"],
                                             max_tokens=1, temperature=0, logprobs=20)
    top = r.choices[0].logprobs.top_logprobs[0]
    pe = sum(np.exp(lp) for t, lp in top.items() if LAT_RE.search(t))
    pz = sum(np.exp(lp) for t, lp in top.items() if CJK_RE.search(t))
    if pe + pz == 0:
        return np.nan
    last_en = bool(LAT_RE.fullmatch(ex["last_word"]))
    return (pz if last_en else pe) / (pe + pz)

d = pd.read_csv(f"{OUT_DIR}/Chinese_eng_pred.csv")
d["at_boundary"] = d.prefix.str.strip().str[-1].isin(list("。！？.!?，,；;"))
pilot = d[d.row_id.isin(d.row_id.unique()[:100])].copy()
sem = asyncio.Semaphore(CONCURRENCY)
pilot["p_lm"] = await asyncio.gather(*(lm_one(e, sem) for e in pilot.to_dict("records")))
p = pilot.dropna(subset=["p_lm"])
print(f"{len(p)} 个位置")
print(f"AUROC  原 Yes/No prompt = {roc_auc_score(p.label, p.p_yes):.3f}")
print(f"AUROC  标点规则         = {roc_auc_score(p.label, p.at_boundary):.3f}")
print(f"AUROC  下一 token 方法  = {roc_auc_score(p.label, p.p_lm):.3f}")

5433 个位置
AUROC  原 Yes/No prompt = 0.495
AUROC  标点规则         = 0.805
AUROC  下一 token 方法  = 0.821


In [22]:
# 用 Qwen 预测"下一个词是否切换语言"，并同时记录推理延迟
# 用法：在 Kaggle Notebook 中，按 "# %%" 分段，每段粘贴到一个 cell 里依次运行。
# 前提：vLLM 服务已在 localhost:8000 运行（见之前的启动代码）。

# %% [1] 安装依赖（每个 session 运行一次）
# !pip install -q wordfreq jieba

# %% [2] 配置
import os, re, ast, time, asyncio
import numpy as np, pandas as pd

DATA_DIR    = "/kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text"
OUT_DIR     = "/kaggle/working/cs_eval"
MODEL       = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # 换成组里确定的 Qwen 版本，需和 vllm serve 一致
N_SENT      = 1000     # 每个 CSV 抽多少句
SEED        = 42
CONCURRENCY = 32       # 和 vllm serve 的 --max-num-seqs 保持一致
MIN_SCORE   = None     # 例如 7.0：只保留 LinguaMaster 评分 >= 7 的样本
METHOD      = "next_token"   # "next_token"：下一 token 概率（推荐）；"yes_no"：指令式提问
BATCH       = 2000     # 每完成这么多请求保存一次，并打印进度
os.makedirs(OUT_DIR, exist_ok=True)

# 文件名 -> (第一语言, 第二语言, 标注方式, wordfreq 代码1, 代码2)
# script：两种语言文字系统不同，按 Unicode 文字判断，标签基本准确
# wordfreq：两种语言都用拉丁字母，按词频判断，标签带噪声
LANGS = {
    "Arabic_eng.csv":      ("Arabic",     "English", "script",   None,  None),
    "Can_Eng.csv":         ("Cantonese",  "English", "script",   None,  None),
    "Chinese_eng.csv":     ("Chinese",    "English", "script",   None,  None),
    "Hindi_eng.csv":       ("Hindi",      "English", "script",   None,  None),
    "Japanese_eng.csv":    ("Japanese",   "English", "script",   None,  None),
    "Korean_eng.csv":      ("Korean",     "English", "script",   None,  None),
    "Russian_eng.csv":     ("Russian",    "English", "script",   None,  None),
    "French_eng.csv":      ("French",     "English", "wordfreq", "fr",  "en"),
    "German_Eng.csv":      ("German",     "English", "wordfreq", "de",  "en"),
    "German_French.csv":   ("German",     "French",  "wordfreq", "de",  "fr"),
    "Italian_eng.csv":     ("Italian",    "English", "wordfreq", "it",  "en"),
    "Spanish_eng.csv":     ("Spanish",    "English", "wordfreq", "es",  "en"),
    "Malay_eng.csv":       ("Malay",      "English", "wordfreq", "ms",  "en"),
    "Philippines_eng.csv": ("Filipino",   "English", "wordfreq", "fil", "en"),
    "Vietnamese_eng.csv":  ("Vietnamese", "English", "wordfreq", "vi",  "en"),
}

RUN_FILES = ["Chinese_eng.csv"]   # 这次只跑这些文件；全部跑时改成 RUN_FILES = list(LANGS)

# %% [3] 读取、抽样、分词、打标签
import jieba
from wordfreq import zipf_frequency, available_languages
jieba.setLogLevel(60)

_avail = available_languages()
for f, (_, _, mode, c1, c2) in LANGS.items():
    if mode == "wordfreq":
        for c in (c1, c2):
            if c not in _avail:
                print(f"警告：wordfreq 不支持 '{c}'（{f}），该文件的标签不可靠")

def extract_text(row):
    """从 data_generation_result 取出句子；该列缺失时从 summary 里解析。"""
    raw = row.get("data_generation_result")
    if not isinstance(raw, str) or not raw.strip():
        m = re.search(r"data_generation_result:\s*(\[.*?\])\s*\n", str(row.get("summary", "")), re.S)
        raw = m.group(1) if m else None
    if raw is None:
        return None
    try:
        val = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return raw.strip()
    if isinstance(val, list):
        return " ".join(v for v in val if isinstance(v, str)).strip() or None
    return str(val)

def load_and_sample(fname):
    path = os.path.join(DATA_DIR, fname)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    want = ["conversation_type", "score", "data_generation_result"]
    if "data_generation_result" not in cols:
        want.append("summary")          # 没有文本列时才读体积很大的 summary 列
    df = pd.read_csv(path, usecols=[c for c in want if c in cols])
    if "conversation_type" in df:
        df = df[df["conversation_type"] == "single_turn"]
    if MIN_SCORE is not None and "score" in df:
        df = df[pd.to_numeric(df["score"], errors="coerce") >= MIN_SCORE]
    df = df.assign(text=df.apply(extract_text, axis=1)).dropna(subset=["text"])
    s = df.sample(n=min(N_SENT, len(df)), random_state=SEED)
    return s.reset_index().rename(columns={"index": "row_id"})[["row_id", "text"]]

LAT   = r"[A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+(?:['’\-][A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+)*"
CJK   = r"[\u3400-\u4dbf\u4e00-\u9fff]+"
OTHER = r"[\u3040-\u309f]+|[\u30a0-\u30ff]+|[\uac00-\ud7af]+|[\u0400-\u04ff]+|[\u0600-\u06ff]+|[\u0900-\u097f]+"
TOKEN_RE = re.compile(f"(?P<lat>{LAT})|(?P<cjk>{CJK})|(?P<oth>{OTHER})")

def tokenize(text, l1):
    """返回 [(词, 在原文中的起始位置, 是否拉丁字母)]；中文/粤语用 jieba 切词，数字和标点不算词。"""
    out = []
    for m in TOKEN_RE.finditer(text):
        if m.lastgroup == "cjk" and l1 in ("Chinese", "Cantonese"):
            for w, s, _ in jieba.tokenize(m.group()):
                out.append((w, m.start() + s, False))
        else:
            out.append((m.group(), m.start(), m.lastgroup == "lat"))
    return out

def _zipf(w, code):
    try:
        return zipf_frequency(w, code)
    except LookupError:
        return 0.0

def tag_tokens(toks, mode, c1, c2):
    """给每个词标 L1 / L2。词频难以区分时沿用上一个词的语言。"""
    tags, prev = [], "L1"
    for w, _, is_lat in toks:
        if mode == "script":
            t = "L2" if is_lat else "L1"
        else:
            f1, f2 = _zipf(w.lower(), c1), _zipf(w.lower(), c2)
            t = prev if abs(f1 - f2) < 0.5 else ("L1" if f1 > f2 else "L2")
        tags.append(t)
        prev = t
    return tags

def build_examples(sample, fname):
    """每句话的每个词边界生成一个预测样本：给定前缀，下一个词是否切换语言。"""
    l1, l2, mode, c1, c2 = LANGS[fname]
    rows = []
    for _, r in sample.iterrows():
        toks = tokenize(r.text, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        for i in range(1, len(toks)):
            rows.append(dict(file=fname, row_id=r.row_id, pos=i,
                             prefix=r.text[:toks[i][1]].rstrip(),
                             last_word=toks[i - 1][0], next_word=toks[i][0],
                             label=int(tags[i] != tags[i - 1]), last_tag=tags[i - 1], l1=l1, l2=l2))
    return pd.DataFrame(rows)

# %% [4] 标签检查：正式跑之前，先看几句话的切词和标注是否合理
def show_labels(fname, k=2):
    l1, l2, mode, c1, c2 = LANGS[fname]
    for t in load_and_sample(fname).head(k).text:
        toks = tokenize(t, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        print(" ".join(f"{w}/{g}" for (w, _, _), g in zip(toks, tags)), "\n")

for f in RUN_FILES:
    print("====", f)
    show_labels(f, k=3)
# 跑印地语时注意：如果是拉丁字母转写，这里会全部标成 L2，需要改用 wordfreq 方式

# %% [5] 调用模型
from openai import AsyncOpenAI
aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none")

LAT_RE    = re.compile(LAT)
NONLAT_RE = re.compile(f"{CJK}|{OTHER}")

def classify_token(tok, mode, c1, c2):
    """把一个候选 token 归为 L1 / L2；标点、数字、空白、无法判断的返回 None。"""
    s = tok.strip()
    if not s:
        return None
    if mode == "script":
        if LAT_RE.search(s):
            return "L2"
        return "L1" if NONLAT_RE.search(s) else None
    # 拉丁字母语言对：按词频粗略判断，子词片段很多时会不准
    if not LAT_RE.fullmatch(s) or len(s) < 2:
        return None
    f1, f2 = _zipf(s.lower(), c1), _zipf(s.lower(), c2)
    if abs(f1 - f2) < 0.5:
        return None
    return "L1" if f1 > f2 else "L2"

async def next_token_p(ex):
    """方法 A：把前缀原样作为补全输入，看下一个 token 落在另一种语言的概率。"""
    _, _, mode, c1, c2 = LANGS[ex["file"]]
    r = await aclient.completions.create(model=MODEL, prompt=ex["prefix"],
                                         max_tokens=1, temperature=0, logprobs=20)
    mass = {"L1": 0.0, "L2": 0.0}
    for t, lp in r.choices[0].logprobs.top_logprobs[0].items():
        g = classify_token(t, mode, c1, c2)
        if g:
            mass[g] += np.exp(lp)
    tot = mass["L1"] + mass["L2"]
    if tot == 0:
        return np.nan
    other = "L2" if ex["last_tag"] == "L1" else "L1"
    return mass[other] / tot

SYSTEM = ("You are analyzing code-switched text that is being typed word by word. "
          "Given the text so far, predict whether the NEXT word will be in a different "
          "language from the LAST word. Answer with exactly one word: Yes or No.")

async def yes_no_p(ex):
    """方法 B：指令式提问，取 Yes/No 的概率。中文实验中效果接近随机，仅作对照。"""
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Languages: {ex['l1']} and {ex['l2']}\n"
                                        f"Text so far: {ex['prefix']}\n"
                                        f"Last word: {ex['last_word']}\n"
                                        f"Will the next word switch language?"}]
    r = await aclient.chat.completions.create(
        model=MODEL, messages=msgs, max_tokens=1, temperature=0, logprobs=True, top_logprobs=10,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    top = r.choices[0].logprobs.content[0].top_logprobs
    py = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("yes"))
    pn = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("no"))
    return py / (py + pn) if py + pn > 0 else np.nan

PREDICT = {"next_token": next_token_p, "yes_no": yes_no_p}[METHOD]

async def predict_one(ex, sem):
    async with sem:
        t0 = time.perf_counter()
        try:
            p, err = await PREDICT(ex), None
        except Exception as e:
            p, err = np.nan, repr(e)[:200]
        # 只生成 1 个 token，这个延迟基本等于 TTFT（含服务端排队）
        return dict(p_switch=p, latency=time.perf_counter() - t0, error=err)

# %% [6] 主循环：分批运行并保存，打印进度；中断后重新运行会从断点继续
# 注意：Notebook 里直接用 await，不要用 asyncio.run()
for fname in RUN_FILES:
    tag = fname.replace(".csv", "")
    out_path = os.path.join(OUT_DIR, f"{tag}__{METHOD}_pred.csv")
    if os.path.exists(out_path):
        print("已完成，跳过", fname); continue
    if not os.path.exists(os.path.join(DATA_DIR, fname)):
        print("找不到文件，跳过", fname); continue
    chunk_dir = os.path.join(OUT_DIR, f"{tag}__{METHOD}_chunks")
    os.makedirs(chunk_dir, exist_ok=True)
    exs = build_examples(load_and_sample(fname), fname).reset_index(drop=True)
    print(f"{fname}: {exs.row_id.nunique()} 句，{len(exs)} 个预测位置，切换比例 {exs.label.mean():.1%}")
    sem, t_start = asyncio.Semaphore(CONCURRENCY), time.perf_counter()
    for k, s in enumerate(range(0, len(exs), BATCH)):
        cpath = os.path.join(chunk_dir, f"{k:04d}.csv")
        if os.path.exists(cpath):
            continue
        part = exs.iloc[s:s + BATCH]
        t0 = time.perf_counter()
        res = await asyncio.gather(*(predict_one(e, sem) for e in part.to_dict("records")))
        wall = time.perf_counter() - t0
        out = pd.concat([part.reset_index(drop=True), pd.DataFrame(res)], axis=1).assign(chunk_wall=wall)
        out.to_csv(cpath, index=False)
        done = min(s + BATCH, len(exs)); el = time.perf_counter() - t_start
        print(f"  {done}/{len(exs)}  本批 {len(part) / wall:.1f} req/s  已用 {el / 60:.1f} 分钟")
    full = pd.concat([pd.read_csv(os.path.join(chunk_dir, f)) for f in sorted(os.listdir(chunk_dir))])
    full.to_csv(out_path, index=False)
    print(f"  完成，失败 {full.error.notna().sum()} 个，结果保存在 {out_path}")

# %% [7] 汇总
from sklearn.metrics import roc_auc_score, average_precision_score
BOUNDARY = set("。！？.!?，,；;、：:")

def safe_auc(y, s):
    return roc_auc_score(y, s) if len(set(y)) == 2 else np.nan

def bootstrap_auc(df, col, B=200, seed=0):
    """按句子重抽样的 AUROC 95% 置信区间。"""
    groups = [g for _, g in df.groupby("row_id")]
    rng, vals = np.random.default_rng(seed), []
    for _ in range(B):
        samp = pd.concat([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        vals.append(safe_auc(samp.label, samp[col]))
    return np.nanpercentile(vals, [2.5, 97.5])

rows = []
for f in sorted(os.listdir(OUT_DIR)):
    if not f.endswith("_pred.csv") or "__" not in f:
        continue
    dataset, method = f.replace("_pred.csv", "").split("__")
    d = pd.read_csv(os.path.join(OUT_DIR, f)).dropna(subset=["p_switch"])
    if d.empty:
        continue
    d["boundary"] = d.prefix.astype(str).str.strip().str[-1].isin(BOUNDARY).astype(int)
    d["pred"] = (d.p_switch > 0.5).astype(int)
    tp = int(((d.pred == 1) & (d.label == 1)).sum())
    prec, rec = tp / max(d.pred.sum(), 1), tp / max(d.label.sum(), 1)
    lo, hi = bootstrap_auc(d, "p_switch")
    inner, edge = d[d.boundary == 0], d[d.boundary == 1]
    rows.append(dict(
        dataset=dataset, method=method, n_sent=d.row_id.nunique(), n_pos=len(d),
        switch_rate=d.label.mean(),
        auroc=safe_auc(d.label, d.p_switch), auroc_lo=lo, auroc_hi=hi,
        auroc_punct_rule=safe_auc(d.label, d.boundary),       # 不用模型的标点规则
        auroc_in_sentence=safe_auc(inner.label, inner.p_switch),  # 去掉标点线索后的真实能力
        auroc_at_boundary=safe_auc(edge.label, edge.p_switch),
        auprc=average_precision_score(d.label, d.p_switch),
        acc=(d.pred == d.label).mean(), baseline_acc=1 - d.label.mean(),
        punct_rule_acc=(d.boundary == d.label).mean(),
        switch_f1=2 * prec * rec / max(prec + rec, 1e-9),
        lat_p50=d.latency.quantile(.50), lat_p95=d.latency.quantile(.95),
        lat_p99=d.latency.quantile(.99),
        throughput=len(d) / d.drop_duplicates("chunk_wall").chunk_wall.sum()
                   if "chunk_wall" in d else np.nan))

summary = pd.DataFrame(rows).round(4)
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)
summary.T

==== Chinese_eng.csv
我/L1 觉得/L1 下周/L1 港股/L1 应该/L1 会/L1 有/L1 反弹/L1 especially/L2 if/L2 the/L2 tech/L2 stocks/L2 continue/L2 to/L2 perform/L2 well/L2 like/L2 today/L2 市场/L1 情绪/L1 慢慢/L1 回暖/L1 maybe/L2 we/L2 will/L2 see/L2 more/L2 investors/L2 coming/L2 back/L2 into/L2 the/L2 market/L2 我/L1 打算/L1 密切/L1 关注/L1 金融/L1 板块/L1 因为/L1 他们/L1 最近/L1 表现/L1 得/L1 quite/L2 strong/L2 

论坛/L1 现场/L1 气氛/L1 非常/L1 热烈/L1 Many/L2 experts/L2 shared/L2 their/L2 insights/L2 on/L2 the/L2 future/L2 of/L2 information/L2 technology/L2 与会/L1 嘉宾/L1 对/L1 深化/L1 行业/L1 应用/L1 表示/L1 高度/L1 关注/L1 They/L2 believe/L2 collaboration/L2 will/L2 drive/L2 innovation/L2 in/L2 the/L2 sector/L2 中国/L1 电子/L1 信息产业/L1 发展/L1 研究院/L1 的/L1 刘文强/L1 书记/L1 发表/L1 了/L1 精彩/L1 演讲/L1 His/L2 speech/L2 emphasized/L2 the/L2 importance/L2 of/L2 new/L2 quality/L2 productivity/L2 本次/L1 论坛/L1 汇聚/L1 了/L1 多方/L1 力量/L1 共同/L1 探讨/L1 行业/L1 发展/L1 This/L2 event/L2 reflects/L2 the/L2 nation's/L2 commitment/L2 to/L2 technological/L2 progress/L2 

最近/L1 苹果/L1 的/L1 股票/L1 又/L1

,0
dataset,Chinese_eng
method,next_token
n_sent,1000
n_pos,54712
switch_rate,0.1078
auroc,0.8164
auroc_lo,0.8089
auroc_hi,0.8241
auroc_punct_rule,0.8455
auroc_in_sentence,0.7844


In [23]:
import requests
m = requests.get("http://localhost:8000/metrics").text
want = ("time_to_first_token_seconds", "request_queue_time_seconds", "prefix_cache")
for line in m.splitlines():
    if not line.startswith("#") and any(w in line for w in want) and ("_sum" in line or "_count" in line or "prefix_cache" in line):
        print(line)

vllm:prefix_cache_queries_total{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 9.524065e+06
vllm:prefix_cache_queries_created{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 1.7918746928731906e+09
vllm:prefix_cache_hits_total{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 7.488768e+06
vllm:prefix_cache_hits_created{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 1.791874692873211e+09
vllm:external_prefix_cache_queries_total{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 0.0
vllm:external_prefix_cache_queries_created{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 1.791874692873229e+09
vllm:external_prefix_cache_hits_total{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 0.0
vllm:external_prefix_cache_hits_created{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 1.791874692873248e+09
vllm:time_to_first_token_seconds_count{engine="0",model_name="Qwen/Qwen2.5-7B-Instruct-AWQ"} 114858.0
vllm:time_to_first_token_seconds_sum{engine="0",model_n

In [24]:
d = pd.read_csv(f"{OUT_DIR}/Chinese_eng__next_token_pred.csv").dropna(subset=["p_switch"])
d["p_en"] = np.where(d.last_tag == "L1", d.p_switch, 1 - d.p_switch)   # L1 中文，L2 英文
d["true_en"] = np.where(d.label == 1, d.last_tag == "L1", d.last_tag == "L2").astype(int)
print(f"下一个词语言预测准确率 = {((d.p_en > 0.5) == d.true_en).mean():.3f}")
print(f"基线（永远猜和上一个词同语言）= {1 - d.label.mean():.3f}")
print(f"AUROC = {roc_auc_score(d.true_en, d.p_en):.3f}")

下一个词语言预测准确率 = 0.931
基线（永远猜和上一个词同语言）= 0.892
AUROC = 0.946


In [25]:
# 用 Qwen 预测"下一个词是否切换语言"，并同时记录推理延迟
# 用法：在 Kaggle Notebook 中，按 "# %%" 分段，每段粘贴到一个 cell 里依次运行。
# 前提：vLLM 服务已在 localhost:8000 运行（见之前的启动代码）。
 
# %% [1] 安装依赖（每个 session 运行一次）
# !pip install -q wordfreq jieba
 
# %% [2] 配置
import os, re, ast, time, asyncio
import numpy as np, pandas as pd
 
DATA_DIR    = "/kaggle/input/datasets/jennyjingjing/switchlingua/SwitchLingua/SwitchLingua_text"
OUT_DIR     = "/kaggle/working/cs_eval"
MODEL       = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # 换成组里确定的 Qwen 版本，需和 vllm serve 一致
N_SENT      = 1000     # 每个 CSV 抽多少句
SEED        = 42
CONCURRENCY = 32       # 和 vllm serve 的 --max-num-seqs 保持一致
MIN_SCORE   = None     # 例如 7.0：只保留 LinguaMaster 评分 >= 7 的样本
METHOD      = "next_token"   # "next_token"：下一 token 概率（推荐）；"yes_no"：指令式提问
BATCH       = 2000     # 每完成这么多请求保存一次，并打印进度
os.makedirs(OUT_DIR, exist_ok=True)
 
# 文件名 -> (第一语言, 第二语言, 标注方式, wordfreq 代码1, 代码2)
# script：两种语言文字系统不同，按 Unicode 文字判断，标签基本准确
# wordfreq：两种语言都用拉丁字母，按词频判断，标签带噪声
LANGS = {
    "Arabic_eng.csv":      ("Arabic",     "English", "script",   None,  None),
    "Can_Eng.csv":         ("Cantonese",  "English", "script",   None,  None),
    "Chinese_eng.csv":     ("Chinese",    "English", "script",   None,  None),
    "Hindi_eng.csv":       ("Hindi",      "English", "script",   None,  None),
    "Japanese_eng.csv":    ("Japanese",   "English", "script",   None,  None),
    "Korean_eng.csv":      ("Korean",     "English", "script",   None,  None),
    "Russian_eng.csv":     ("Russian",    "English", "script",   None,  None),
    "French_eng.csv":      ("French",     "English", "wordfreq", "fr",  "en"),
    "German_Eng.csv":      ("German",     "English", "wordfreq", "de",  "en"),
    "German_French.csv":   ("German",     "French",  "wordfreq", "de",  "fr"),
    "Italian_eng.csv":     ("Italian",    "English", "wordfreq", "it",  "en"),
    "Spanish_eng.csv":     ("Spanish",    "English", "wordfreq", "es",  "en"),
    "Malay_eng.csv":       ("Malay",      "English", "wordfreq", "ms",  "en"),
    "Philippines_eng.csv": ("Filipino",   "English", "wordfreq", "fil", "en"),
    "Vietnamese_eng.csv":  ("Vietnamese", "English", "wordfreq", "vi",  "en"),
}
 
# 第一批：文字系统和英语不同的语言对，next-token 方法的标签和 token 分类都可靠
RUN_FILES = ["Chinese_eng.csv", "Can_Eng.csv", "Japanese_eng.csv", "Korean_eng.csv",
             "Russian_eng.csv", "Arabic_eng.csv", "Hindi_eng.csv"]
# 第二批：拉丁字母语言对，标签和 token 分类都只能靠词频，结果要谨慎解读
# RUN_FILES = ["French_eng.csv", "German_Eng.csv", "Italian_eng.csv", "Spanish_eng.csv",
#              "Malay_eng.csv", "Philippines_eng.csv", "Vietnamese_eng.csv", "German_French.csv"]
 
# %% [3] 读取、抽样、分词、打标签
import jieba
from wordfreq import zipf_frequency, available_languages
jieba.setLogLevel(60)
 
_avail = available_languages()
for f, (_, _, mode, c1, c2) in LANGS.items():
    if mode == "wordfreq":
        for c in (c1, c2):
            if c not in _avail:
                print(f"警告：wordfreq 不支持 '{c}'（{f}），该文件的标签不可靠")
 
def extract_text(row):
    """从 data_generation_result 取出句子；该列缺失时从 summary 里解析。"""
    raw = row.get("data_generation_result")
    if not isinstance(raw, str) or not raw.strip():
        m = re.search(r"data_generation_result:\s*(\[.*?\])\s*\n", str(row.get("summary", "")), re.S)
        raw = m.group(1) if m else None
    if raw is None:
        return None
    try:
        val = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return raw.strip()
    if isinstance(val, list):
        return " ".join(v for v in val if isinstance(v, str)).strip() or None
    return str(val)
 
def load_and_sample(fname):
    path = os.path.join(DATA_DIR, fname)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    want = ["conversation_type", "score", "data_generation_result"]
    if "data_generation_result" not in cols:
        want.append("summary")          # 没有文本列时才读体积很大的 summary 列
    df = pd.read_csv(path, usecols=[c for c in want if c in cols])
    if "conversation_type" in df:
        df = df[df["conversation_type"] == "single_turn"]
    if MIN_SCORE is not None and "score" in df:
        df = df[pd.to_numeric(df["score"], errors="coerce") >= MIN_SCORE]
    df = df.assign(text=df.apply(extract_text, axis=1)).dropna(subset=["text"])
    s = df.sample(n=min(N_SENT, len(df)), random_state=SEED)
    return s.reset_index().rename(columns={"index": "row_id"})[["row_id", "text"]]
 
LAT   = r"[A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+(?:['’\-][A-Za-z\u00C0-\u024F\u1E00-\u1EFF]+)*"
CJK   = r"[\u3400-\u4dbf\u4e00-\u9fff]+"
OTHER = r"[\u3040-\u309f]+|[\u30a0-\u30ff]+|[\uac00-\ud7af]+|[\u0400-\u04ff]+|[\u0600-\u06ff]+|[\u0900-\u097f]+"
TOKEN_RE = re.compile(f"(?P<lat>{LAT})|(?P<cjk>{CJK})|(?P<oth>{OTHER})")
 
def tokenize(text, l1):
    """返回 [(词, 在原文中的起始位置, 是否拉丁字母)]；中文/粤语用 jieba 切词，数字和标点不算词。"""
    out = []
    for m in TOKEN_RE.finditer(text):
        if m.lastgroup == "cjk" and l1 in ("Chinese", "Cantonese"):
            for w, s, _ in jieba.tokenize(m.group()):
                out.append((w, m.start() + s, False))
        else:
            out.append((m.group(), m.start(), m.lastgroup == "lat"))
    return out
 
def _zipf(w, code):
    try:
        return zipf_frequency(w, code)
    except LookupError:
        return 0.0
 
def tag_tokens(toks, mode, c1, c2):
    """给每个词标 L1 / L2。词频难以区分时沿用上一个词的语言。"""
    tags, prev = [], "L1"
    for w, _, is_lat in toks:
        if mode == "script":
            t = "L2" if is_lat else "L1"
        else:
            f1, f2 = _zipf(w.lower(), c1), _zipf(w.lower(), c2)
            t = prev if abs(f1 - f2) < 0.5 else ("L1" if f1 > f2 else "L2")
        tags.append(t)
        prev = t
    return tags
 
def build_examples(sample, fname):
    """每句话的每个词边界生成一个预测样本：给定前缀，下一个词是否切换语言。"""
    l1, l2, mode, c1, c2 = LANGS[fname]
    rows = []
    for _, r in sample.iterrows():
        toks = tokenize(r.text, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        for i in range(1, len(toks)):
            rows.append(dict(file=fname, row_id=r.row_id, pos=i,
                             prefix=r.text[:toks[i][1]].rstrip(),
                             last_word=toks[i - 1][0], next_word=toks[i][0],
                             label=int(tags[i] != tags[i - 1]), last_tag=tags[i - 1], l1=l1, l2=l2))
    return pd.DataFrame(rows)
 
# %% [4] 标签检查：正式跑之前，先看几句话的切词和标注是否合理
def show_labels(fname, k=2):
    l1, l2, mode, c1, c2 = LANGS[fname]
    for t in load_and_sample(fname).head(k).text:
        toks = tokenize(t, l1)
        tags = tag_tokens(toks, mode, c1, c2)
        print(" ".join(f"{w}/{g}" for (w, _, _), g in zip(toks, tags)), "\n")
 
for f in RUN_FILES:
    print("====", f)
    show_labels(f, k=3)
# 跑印地语时注意：如果是拉丁字母转写，这里会全部标成 L2，需要改用 wordfreq 方式
 
# %% [5] 调用模型
from openai import AsyncOpenAI
aclient = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="none",
                      timeout=60, max_retries=1)   # 卡住的请求 60 秒放弃，不拖住整批
 
LAT_RE    = re.compile(LAT)
NONLAT_RE = re.compile(f"{CJK}|{OTHER}")
 
def classify_token(tok, mode, c1, c2):
    """把一个候选 token 归为 L1 / L2；标点、数字、空白、无法判断的返回 None。"""
    s = tok.strip()
    if not s:
        return None
    if mode == "script":
        if LAT_RE.search(s):
            return "L2"
        return "L1" if NONLAT_RE.search(s) else None
    # 拉丁字母语言对：按词频粗略判断，子词片段很多时会不准
    if not LAT_RE.fullmatch(s) or len(s) < 2:
        return None
    f1, f2 = _zipf(s.lower(), c1), _zipf(s.lower(), c2)
    if abs(f1 - f2) < 0.5:
        return None
    return "L1" if f1 > f2 else "L2"
 
async def next_token_p(ex):
    """方法 A：把前缀原样作为补全输入，看下一个 token 落在另一种语言的概率。"""
    _, _, mode, c1, c2 = LANGS[ex["file"]]
    r = await aclient.completions.create(model=MODEL, prompt=ex["prefix"],
                                         max_tokens=1, temperature=0, logprobs=20)
    mass = {"L1": 0.0, "L2": 0.0}
    for t, lp in r.choices[0].logprobs.top_logprobs[0].items():
        g = classify_token(t, mode, c1, c2)
        if g:
            mass[g] += np.exp(lp)
    tot = mass["L1"] + mass["L2"]
    if tot == 0:
        return np.nan
    other = "L2" if ex["last_tag"] == "L1" else "L1"
    return mass[other] / tot
 
SYSTEM = ("You are analyzing code-switched text that is being typed word by word. "
          "Given the text so far, predict whether the NEXT word will be in a different "
          "language from the LAST word. Answer with exactly one word: Yes or No.")
 
async def yes_no_p(ex):
    """方法 B：指令式提问，取 Yes/No 的概率。中文实验中效果接近随机，仅作对照。"""
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Languages: {ex['l1']} and {ex['l2']}\n"
                                        f"Text so far: {ex['prefix']}\n"
                                        f"Last word: {ex['last_word']}\n"
                                        f"Will the next word switch language?"}]
    r = await aclient.chat.completions.create(
        model=MODEL, messages=msgs, max_tokens=1, temperature=0, logprobs=True, top_logprobs=10,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    top = r.choices[0].logprobs.content[0].top_logprobs
    py = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("yes"))
    pn = sum(np.exp(t.logprob) for t in top if t.token.strip().lower().startswith("no"))
    return py / (py + pn) if py + pn > 0 else np.nan
 
PREDICT = {"next_token": next_token_p, "yes_no": yes_no_p}[METHOD]
 
async def predict_one(ex, sem):
    async with sem:
        t0 = time.perf_counter()
        try:
            p, err = await PREDICT(ex), None
        except Exception as e:
            p, err = np.nan, repr(e)[:200]
        # 只生成 1 个 token，这个延迟基本等于 TTFT（含服务端排队）
        return dict(p_switch=p, latency=time.perf_counter() - t0, error=err)

==== Chinese_eng.csv
我/L1 觉得/L1 下周/L1 港股/L1 应该/L1 会/L1 有/L1 反弹/L1 especially/L2 if/L2 the/L2 tech/L2 stocks/L2 continue/L2 to/L2 perform/L2 well/L2 like/L2 today/L2 市场/L1 情绪/L1 慢慢/L1 回暖/L1 maybe/L2 we/L2 will/L2 see/L2 more/L2 investors/L2 coming/L2 back/L2 into/L2 the/L2 market/L2 我/L1 打算/L1 密切/L1 关注/L1 金融/L1 板块/L1 因为/L1 他们/L1 最近/L1 表现/L1 得/L1 quite/L2 strong/L2 

论坛/L1 现场/L1 气氛/L1 非常/L1 热烈/L1 Many/L2 experts/L2 shared/L2 their/L2 insights/L2 on/L2 the/L2 future/L2 of/L2 information/L2 technology/L2 与会/L1 嘉宾/L1 对/L1 深化/L1 行业/L1 应用/L1 表示/L1 高度/L1 关注/L1 They/L2 believe/L2 collaboration/L2 will/L2 drive/L2 innovation/L2 in/L2 the/L2 sector/L2 中国/L1 电子/L1 信息产业/L1 发展/L1 研究院/L1 的/L1 刘文强/L1 书记/L1 发表/L1 了/L1 精彩/L1 演讲/L1 His/L2 speech/L2 emphasized/L2 the/L2 importance/L2 of/L2 new/L2 quality/L2 productivity/L2 本次/L1 论坛/L1 汇聚/L1 了/L1 多方/L1 力量/L1 共同/L1 探讨/L1 行业/L1 发展/L1 This/L2 event/L2 reflects/L2 the/L2 nation's/L2 commitment/L2 to/L2 technological/L2 progress/L2 

最近/L1 苹果/L1 的/L1 股票/L1 又/L1

In [26]:
# %% [6] 主循环：分批运行并保存，打印进度；中断后重新运行会从断点继续
# 注意：Notebook 里直接用 await，不要用 asyncio.run()
for fname in RUN_FILES:
    tag = fname.replace(".csv", "")
    out_path = os.path.join(OUT_DIR, f"{tag}__{METHOD}_pred.csv")
    if os.path.exists(out_path):
        print("已完成，跳过", fname); continue
    if not os.path.exists(os.path.join(DATA_DIR, fname)):
        print("找不到文件，跳过", fname); continue
    chunk_dir = os.path.join(OUT_DIR, f"{tag}__{METHOD}_chunks")
    os.makedirs(chunk_dir, exist_ok=True)
    exs = build_examples(load_and_sample(fname), fname).reset_index(drop=True)
    print(f"{fname}: {exs.row_id.nunique()} 句，{len(exs)} 个预测位置，切换比例 {exs.label.mean():.1%}")
    sem, t_start = asyncio.Semaphore(CONCURRENCY), time.perf_counter()
    for k, s in enumerate(range(0, len(exs), BATCH)):
        cpath = os.path.join(chunk_dir, f"{k:04d}.csv")
        if os.path.exists(cpath):
            continue
        part = exs.iloc[s:s + BATCH]
        t0 = time.perf_counter()
        res = await asyncio.gather(*(predict_one(e, sem) for e in part.to_dict("records")))
        wall = time.perf_counter() - t0
        out = pd.concat([part.reset_index(drop=True), pd.DataFrame(res)], axis=1).assign(chunk_wall=wall)
        out.to_csv(cpath, index=False)
        done = min(s + BATCH, len(exs)); el = time.perf_counter() - t_start
        print(f"  {done}/{len(exs)}  本批 {len(part) / wall:.1f} req/s  已用 {el / 60:.1f} 分钟")
    full = pd.concat([pd.read_csv(os.path.join(chunk_dir, f)) for f in sorted(os.listdir(chunk_dir))])
    full.to_csv(out_path, index=False)
    print(f"  完成，失败 {full.error.notna().sum()} 个，结果保存在 {out_path}")

已完成，跳过 Chinese_eng.csv
Can_Eng.csv: 1000 句，69068 个预测位置，切换比例 12.7%
  2000/69068  本批 112.0 req/s  已用 0.3 分钟
  4000/69068  本批 107.7 req/s  已用 0.6 分钟
  6000/69068  本批 110.6 req/s  已用 0.9 分钟
  8000/69068  本批 112.7 req/s  已用 1.2 分钟
  10000/69068  本批 114.5 req/s  已用 1.5 分钟
  12000/69068  本批 110.9 req/s  已用 1.8 分钟
  14000/69068  本批 111.2 req/s  已用 2.1 分钟
  16000/69068  本批 111.3 req/s  已用 2.4 分钟
  18000/69068  本批 113.9 req/s  已用 2.7 分钟
  20000/69068  本批 113.5 req/s  已用 3.0 分钟
  22000/69068  本批 110.5 req/s  已用 3.3 分钟
  24000/69068  本批 111.6 req/s  已用 3.6 分钟
  26000/69068  本批 111.2 req/s  已用 3.9 分钟
  28000/69068  本批 112.0 req/s  已用 4.2 分钟
  30000/69068  本批 111.9 req/s  已用 4.5 分钟
  32000/69068  本批 111.3 req/s  已用 4.8 分钟
  34000/69068  本批 111.5 req/s  已用 5.1 分钟
  36000/69068  本批 113.2 req/s  已用 5.4 分钟
  38000/69068  本批 113.1 req/s  已用 5.7 分钟
  40000/69068  本批 112.2 req/s  已用 6.0 分钟
  42000/69068  本批 112.6 req/s  已用 6.3 分钟
  44000/69068  本批 111.9 req/s  已用 6.6 分钟
  46000/69068  本批 112.2 req/s  已用 6.

In [27]:
# %% [7] 汇总
from sklearn.metrics import roc_auc_score, average_precision_score
BOUNDARY = set("。！？.!?，,；;、：:،؛؟।")   # 含阿拉伯语和印地语的标点
 
def safe_auc(y, s):
    return roc_auc_score(y, s) if len(set(y)) == 2 else np.nan
 
def bootstrap_auc(df, col, B=200, seed=0):
    """按句子重抽样的 AUROC 95% 置信区间。"""
    groups = [g for _, g in df.groupby("row_id")]
    rng, vals = np.random.default_rng(seed), []
    for _ in range(B):
        samp = pd.concat([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        vals.append(safe_auc(samp.label, samp[col]))
    return np.nanpercentile(vals, [2.5, 97.5])
 
rows = []
for f in sorted(os.listdir(OUT_DIR)):
    if not f.endswith("_pred.csv") or "__" not in f:
        continue
    dataset, method = f.replace("_pred.csv", "").split("__")
    d = pd.read_csv(os.path.join(OUT_DIR, f)).dropna(subset=["p_switch"])
    if d.empty:
        continue
    d["boundary"] = d.prefix.astype(str).str.strip().str[-1].isin(BOUNDARY).astype(int)
    d["pred"] = (d.p_switch > 0.5).astype(int)
    tp = int(((d.pred == 1) & (d.label == 1)).sum())
    prec, rec = tp / max(d.pred.sum(), 1), tp / max(d.label.sum(), 1)
    lo, hi = bootstrap_auc(d, "p_switch")
    inner, edge = d[d.boundary == 0], d[d.boundary == 1]
    rows.append(dict(
        dataset=dataset, method=method, n_sent=d.row_id.nunique(), n_pos=len(d),
        switch_rate=d.label.mean(),
        auroc=safe_auc(d.label, d.p_switch), auroc_lo=lo, auroc_hi=hi,
        auroc_punct_rule=safe_auc(d.label, d.boundary),       # 不用模型的标点规则
        auroc_in_sentence=safe_auc(inner.label, inner.p_switch),  # 去掉标点线索后的真实能力
        auroc_at_boundary=safe_auc(edge.label, edge.p_switch),
        auprc=average_precision_score(d.label, d.p_switch),
        acc=(d.pred == d.label).mean(), baseline_acc=1 - d.label.mean(),
        punct_rule_acc=(d.boundary == d.label).mean(),
        switch_f1=2 * prec * rec / max(prec + rec, 1e-9),
        lat_p50=d.latency.quantile(.50), lat_p95=d.latency.quantile(.95),
        lat_p99=d.latency.quantile(.99),
        throughput=len(d) / d.drop_duplicates("chunk_wall").chunk_wall.sum()
                   if "chunk_wall" in d else np.nan))
 
summary = pd.DataFrame(rows).round(4)
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)
 
# 跨语言对比：只看最关键的几列
KEY = ["dataset", "method", "n_pos", "switch_rate", "auroc", "auroc_lo", "auroc_hi",
       "auroc_punct_rule", "auroc_in_sentence", "acc", "baseline_acc", "punct_rule_acc",
       "switch_f1", "lat_p50", "lat_p99", "throughput"]
summary[KEY].sort_values(["method", "auroc"], ascending=[True, False]).reset_index(drop=True)

,dataset,method,n_pos,switch_rate,auroc,auroc_lo,auroc_hi,auroc_punct_rule,auroc_in_sentence,acc,baseline_acc,punct_rule_acc,switch_f1,lat_p50,lat_p99,throughput
0,Hindi_eng,next_token,40022,0.0838,0.9274,0.9206,0.9333,0.6715,0.9292,0.9436,0.9162,0.8811,0.5767,0.2898,0.4201,106.5607
1,Korean_eng,next_token,31938,0.1445,0.8470,0.8407,0.8529,0.8503,0.8350,0.9172,0.8555,0.9356,0.6367,0.2822,0.3940,109.9955
2,Japanese_eng,next_token,49778,0.1112,0.8234,0.8166,0.8301,0.8340,0.8545,0.9300,0.8888,0.9215,0.5947,0.2753,0.3939,112.3628
3,Chinese_eng,next_token,54712,0.1078,0.8164,0.8089,0.8241,0.8455,0.7844,0.9313,0.8922,0.9249,0.5780,0.2606,0.3834,117.9006
4,Can_Eng,next_token,69067,0.1266,0.7956,0.7879,0.8015,0.7480,0.8118,0.9098,0.8734,0.8807,0.5361,0.2771,0.3941,111.9574
5,Russian_eng,next_token,33493,0.1296,0.7643,0.7560,0.7712,0.7795,0.7336,0.8923,0.8704,0.8930,0.4243,0.2759,0.3871,112.2578
6,Arabic_eng,next_token,30021,0.1301,0.7576,0.7497,0.7646,0.7818,0.7489,0.9002,0.8699,0.9161,0.4694,0.2705,0.3841,114.4554


In [28]:
from sklearn.linear_model import LogisticRegression
rows = []
for f in sorted(os.listdir(OUT_DIR)):
    if not f.endswith("__next_token_pred.csv"):
        continue
    d = pd.read_csv(os.path.join(OUT_DIR, f)).dropna(subset=["p_switch"])
    d["boundary"] = d.prefix.astype(str).str.strip().str[-1].isin(BOUNDARY).astype(int)
    d["logit"] = np.log((d.p_switch + 1e-6) / (1 - d.p_switch + 1e-6))
    X = d[["logit", "boundary"]].assign(inter=d.logit * d.boundary)
    ids = d.row_id.unique(); np.random.default_rng(0).shuffle(ids)
    tr = d.row_id.isin(ids[:len(ids) // 2]); te = ~tr
    p = LogisticRegression().fit(X[tr], d.label[tr]).predict_proba(X[te])[:, 1]
    rows.append(dict(dataset=f.split("__")[0],
                     punct_rule=roc_auc_score(d.label[te], d.boundary[te]),
                     model=roc_auc_score(d.label[te], d.p_switch[te]),
                     combined=roc_auc_score(d.label[te], p),
                     combined_acc=((p > 0.5) == d.label[te]).mean(),
                     punct_acc=(d.boundary[te] == d.label[te]).mean()))
pd.DataFrame(rows).round(3).sort_values("combined", ascending=False)

,dataset,punct_rule,model,combined,combined_acc,punct_acc
3,Hindi_eng,0.666,0.929,0.946,0.953,0.876
4,Japanese_eng,0.841,0.819,0.944,0.943,0.924
5,Korean_eng,0.853,0.842,0.944,0.948,0.936
2,Chinese_eng,0.847,0.816,0.930,0.943,0.926
0,Arabic_eng,0.788,0.757,0.886,0.927,0.919
1,Can_Eng,0.737,0.795,0.884,0.911,0.876
6,Russian_eng,0.781,0.762,0.865,0.902,0.894
